# RSNA 2026 Knee MRI Abnormality Detection: preprocessing and baselines

Part 1 of 2. Data exploration, the preprocessing pipeline, and the first two baselines (fine-tuned ResNet18 on MIP input, text-only classifier). Part 2 (`02_slice_features_and_weak_labels.ipynb`) covers what worked better.

Written for Kaggle Notebooks with the competition data attached (`/kaggle/input/competitions/rsna-knee-abnormality-detection`). Outputs are cleared: the competition data, including radiology reports, must not be redistributed.

## 1. Explore the DICOM data
Check the file layout, series metadata and DICOM properties (pixel spacing, transfer syntax, value range) before building anything.

In [ ]:
import os
print(os.listdir("/kaggle/input/competitions/rsna-knee-abnormality-detection"))


In [ ]:
!pip install pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg gdcm

In [ ]:
import pandas as pd
import pydicom
import os

DATA_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

train_series = pd.read_csv(os.path.join(DATA_ROOT, "train_series.csv"))

# One series per plane/sequence-type combo
sample_series = train_series.groupby(["Anatomical_Plane", "Fluid_Sensitive"]).first().reset_index()

for _, row in sample_series.iterrows():
    series_dir = os.path.join(DATA_ROOT, "train_series", row["StudyInstanceUID"], row["SeriesInstanceUID"])
    if not os.path.isdir(series_dir):
        print(f"Directory not found: {series_dir}")
        continue
    dcm_files = sorted(os.listdir(series_dir))
    print(f"\n--- {row['Anatomical_Plane']} / Fluid_Sensitive={row['Fluid_Sensitive']} ---")
    print(f"Number of slices: {len(dcm_files)}")

    if dcm_files:
        ds = pydicom.dcmread(os.path.join(series_dir, dcm_files[0]))
        print(f"Pixel array shape: {ds.pixel_array.shape}")
        print(f"Pixel spacing: {ds.get('PixelSpacing', 'N/A')}")
        print(f"Bits allocated: {ds.get('BitsAllocated', 'N/A')}")
        print(f"Transfer syntax: {ds.file_meta.TransferSyntaxUID}")
        print(f"Pixel value range: min={ds.pixel_array.min()}, max={ds.pixel_array.max()}")

In [ ]:
!pip install opencv-python-headless

In [ ]:
import os
import pandas as pd
import pydicom
import numpy as np
import cv2

DATA_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

# Auto-generate a real path from the first row of train_series.csv
train_series = pd.read_csv(os.path.join(DATA_ROOT, "train_series.csv"))
sample_study = train_series.iloc[0]["StudyInstanceUID"]
sample_seriesid = train_series.iloc[0]["SeriesInstanceUID"]
sample_dir = os.path.join(DATA_ROOT, "train_series", sample_study, sample_seriesid)
sample_file = os.listdir(sample_dir)[0]

test_dcm = os.path.join(sample_dir, sample_file)
print(f"Testing with: {test_dcm}")

def load_and_resize(dcm_path, target_size=256):
    ds = pydicom.dcmread(dcm_path)
    pixel_array = ds.pixel_array.astype(np.float32)
    resized = cv2.resize(pixel_array, (target_size, target_size), interpolation=cv2.INTER_AREA)
    return resized

resized_img = load_and_resize(test_dcm)
print(f"Resized shape: {resized_img.shape}")
print(f"Value range preserved: min={resized_img.min():.1f}, max={resized_img.max():.1f}")

Here we test the resizing , if it worked and it did.

now we will continue with the normalization process.

## 2. Preprocessing
Per-slice percentile normalization, then a maximum-intensity projection (MIP) per plane, with one series chosen per plane (fluid-sensitive preferred, standard as fallback).

In [ ]:
def normalize_image(img, lower_percentile=1, upper_percentile=99):
    """
    Normalize a single image's pixel intensities to a 0-1 range,
    using percentile clipping to avoid outlier pixels (e.g. bright
    artifacts) from skewing the scale for the whole image.
    """
    lo = np.percentile(img, lower_percentile)
    hi = np.percentile(img, upper_percentile)
    
    # Avoid divide-by-zero on flat/empty images
    if hi - lo < 1e-6:
        return np.zeros_like(img, dtype=np.float32)
    
    clipped = np.clip(img, lo, hi)
    normalized = (clipped - lo) / (hi - lo)
    return normalized.astype(np.float32)


def load_resize_normalize(dcm_path, target_size=256):
    """Full pipeline: load DICOM -> resize -> normalize."""
    ds = pydicom.dcmread(dcm_path)
    pixel_array = ds.pixel_array.astype(np.float32)
    
    resized = cv2.resize(pixel_array, (target_size, target_size), interpolation=cv2.INTER_AREA)
    normalized = normalize_image(resized)
    
    return normalized

# --- Test it on the same file as before ---
result = load_resize_normalize(test_dcm)
print(f"Shape: {result.shape}")
print(f"Value range after normalization: min={result.min():.4f}, max={result.max():.4f}")
print(f"Mean: {result.mean():.4f}, Std: {result.std():.4f}")

In [ ]:
def load_series_mip(series_dir, target_size=256):
    """
    Load all slices in a series, resize+normalize each one individually
    first (to avoid a few huge outlier pixels in one slice distorting
    the whole projection), then take the max value at each pixel
    position across the stack.
    """
    dcm_files = sorted(os.listdir(series_dir))
    
    slices = []
    for f in dcm_files:
        dcm_path = os.path.join(series_dir, f)
        img = load_resize_normalize(dcm_path, target_size=target_size)
        slices.append(img)
    
    stack = np.stack(slices, axis=0)  # shape: (num_slices, H, W)
    mip = np.max(stack, axis=0)       # shape: (H, W)
    
    return mip

# --- Test it on the same series as before ---
sample_dir = os.path.join(DATA_ROOT, "train_series", sample_study, sample_seriesid)
mip_result = load_series_mip(sample_dir)

print(f"MIP shape: {mip_result.shape}")
print(f"Value range: min={mip_result.min():.4f}, max={mip_result.max():.4f}")
print(f"Number of slices used: {len(os.listdir(sample_dir))}")

In [ ]:
PLANES = ["Axial", "Coronal", "Sagittal"]
FLUID_OPTIONS = [0, 1]

def load_study_mips(study_id, train_series_df, target_size=256):
    """
    For a given study, build one MIP per (plane, fluid_sensitive) slot.
    Returns a dict keyed by (plane, fluid_sensitive) -> MIP array,
    plus a fixed-order stacked array for feeding to a model.
    """
    study_rows = train_series_df[train_series_df["StudyInstanceUID"] == study_id]
    
    mips = {}
    for plane in PLANES:
        for fluid in FLUID_OPTIONS:
            matches = study_rows[
                (study_rows["Anatomical_Plane"] == plane) &
                (study_rows["Fluid_Sensitive"] == fluid)
            ]
            
            if len(matches) == 0:
                # Slot missing for this study - fill with zeros as a placeholder
                mips[(plane, fluid)] = np.zeros((target_size, target_size), dtype=np.float32)
                continue
            
            # If multiple series match this slot, just take the first (per our earlier analysis,
            # ~86% of slots have exactly one candidate; duplicates are rare)
            series_id = matches.iloc[0]["SeriesInstanceUID"]
            series_dir = os.path.join(DATA_ROOT, "train_series", study_id, series_id)
            mips[(plane, fluid)] = load_series_mip(series_dir, target_size=target_size)
    
    # Stack in a fixed, consistent order: 6 channels
    stacked = np.stack([mips[(p, f)] for p in PLANES for f in FLUID_OPTIONS], axis=0)
    return stacked, mips

# --- Test on the same sample study ---
stacked_input, mip_dict = load_study_mips(sample_study, train_series)
print(f"Stacked input shape: {stacked_input.shape}")  # should be (6, 256, 256)
for (plane, fluid), mip in mip_dict.items():
    print(f"{plane}/fluid={fluid}: min={mip.min():.3f}, max={mip.max():.3f}")

In [ ]:
# Check how often each of the 6 specific (plane, fluid) slots is missing per study
from itertools import product

combo_grid = list(product(PLANES, FLUID_OPTIONS))
study_ids = train_series["StudyInstanceUID"].unique()

missing_counts = {combo: 0 for combo in combo_grid}

for study_id in study_ids:
    study_rows = train_series[train_series["StudyInstanceUID"] == study_id]
    for plane, fluid in combo_grid:
        matches = study_rows[
            (study_rows["Anatomical_Plane"] == plane) &
            (study_rows["Fluid_Sensitive"] == fluid)
        ]
        if len(matches) == 0:
            missing_counts[(plane, fluid)] += 1

total_studies = len(study_ids)
print(f"Total studies: {total_studies}\n")
for combo, count in missing_counts.items():
    pct = 100 * count / total_studies
    print(f"{combo}: missing in {count} studies ({pct:.1f}%)")

In [ ]:
PLANES = ["Axial", "Coronal", "Sagittal"]

def get_best_series_for_plane(study_rows, plane, preferred_fluid=1):
    """
    Pick the best available series for a given plane, preferring
    fluid-sensitive (fluid=1), falling back to standard (fluid=0)
    if the preferred type isn't present. Returns None if the plane
    is entirely missing (shouldn't happen, since all studies have
    all 3 planes per earlier EDA - but handled defensively anyway).
    """
    plane_rows = study_rows[study_rows["Anatomical_Plane"] == plane]
    if len(plane_rows) == 0:
        return None
    
    preferred = plane_rows[plane_rows["Fluid_Sensitive"] == preferred_fluid]
    if len(preferred) > 0:
        return preferred.iloc[0]["SeriesInstanceUID"]
    
    fallback = plane_rows[plane_rows["Fluid_Sensitive"] != preferred_fluid]
    if len(fallback) > 0:
        return fallback.iloc[0]["SeriesInstanceUID"]
    
    return None  # shouldn't happen, but just in case


def load_study_mips_v2(study_id, train_series_df, target_size=256):
    """
    For a given study, build one MIP per plane (3 total), using the
    best available series per plane rather than a rigid 6-slot grid.
    """
    study_rows = train_series_df[train_series_df["StudyInstanceUID"] == study_id]
    
    mips = {}
    for plane in PLANES:
        series_id = get_best_series_for_plane(study_rows, plane, preferred_fluid=1)
        
        if series_id is None:
            mips[plane] = np.zeros((target_size, target_size), dtype=np.float32)
            continue
        
        series_dir = os.path.join(DATA_ROOT, "train_series", study_id, series_id)
        mips[plane] = load_series_mip(series_dir, target_size=target_size)
    
    # Stack in fixed order: 3 channels
    stacked = np.stack([mips[p] for p in PLANES], axis=0)
    return stacked, mips

# --- Test on the same sample study ---
stacked_input, mip_dict = load_study_mips_v2(sample_study, train_series)
print(f"Stacked input shape: {stacked_input.shape}")  # should be (3, 256, 256)
for plane, mip in mip_dict.items():
    print(f"{plane}: min={mip.min():.3f}, max={mip.max():.3f}")

## 3. Cache the 58 gold-labeled studies
Build a `(58, 3, 256, 256)` array (one MIP per plane) plus the `(58, 12)` label matrix, and cache both so the DICOMs only need decoding once.

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm  # progress bar; pip install tqdm if not available

DATA_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

train_df = pd.read_csv(os.path.join(DATA_ROOT, "train.csv"))
train_series = pd.read_csv(os.path.join(DATA_ROOT, "train_series.csv"))

label_cols = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
              "Medial OA", "Lateral OA", "PF OA", "Effusion",
              "Synovitis", "Baker's", "Contusion", "Fracture"]

# Isolate the 58 gold-labeled studies (non-null labels)
gold_df = train_df[train_df[label_cols].notna().all(axis=1)].reset_index(drop=True)
print(f"Gold-labeled studies found: {len(gold_df)}")

X = []
y = []
failed_studies = []

for _, row in tqdm(gold_df.iterrows(), total=len(gold_df)):
    study_id = row["StudyInstanceUID"]
    try:
        stacked_input, _ = load_study_mips_v2(study_id, train_series)
        X.append(stacked_input)
        y.append(row[label_cols].values.astype(np.float32))
    except Exception as e:
        print(f"Failed on {study_id}: {e}")
        failed_studies.append(study_id)

X = np.stack(X, axis=0)  # shape: (num_studies, 3, 256, 256)
y = np.stack(y, axis=0)  # shape: (num_studies, 12)

print(f"\nFinal dataset shape: X={X.shape}, y={y.shape}")
print(f"Failed studies: {len(failed_studies)}")

# Cache to disk so you never need to rebuild this from raw DICOMs again
np.save("/kaggle/working/gold_X.npy", X)
np.save("/kaggle/working/gold_y.npy", y)
print("\nSaved to /kaggle/working/gold_X.npy and gold_y.npy")

In [ ]:
!pip install iterative-stratification -q

## 4. Baseline: ResNet18 on the MIP input
Fine-tune a pretrained ResNet18 with 12 sigmoid outputs, evaluated with 5-fold multilabel-stratified cross-validation (macro AUC). It overfits heavily with ~45 training studies per fold.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

X = np.load("/kaggle/working/gold_X.npy")  # (58, 3, 256, 256)
y = np.load("/kaggle/working/gold_y.npy")  # (58, 12)

label_cols = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
              "Medial OA", "Lateral OA", "PF OA", "Effusion",
              "Synovitis", "Baker's", "Contusion", "Fracture"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class KneeMRIDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = X
        self.y = y
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = self.X[idx].copy()  # (3, 256, 256)
        label = self.y[idx]

        if self.augment:
            # Simple, cheap augmentations appropriate for medical images
            if np.random.rand() > 0.5:
                img = img[:, :, ::-1].copy()  # horizontal flip
            if np.random.rand() > 0.5:
                k = np.random.choice([1, 2, 3])
                img = np.rot90(img, k=k, axes=(1, 2)).copy()  # small rotations

        return torch.tensor(img, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)


def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, 12)  # 12 output logits, one per finding
    return model.to(device)


def train_one_fold(train_idx, val_idx, X, y, epochs=15, lr=1e-4, batch_size=8):
    train_ds = KneeMRIDataset(X[train_idx], y[train_idx], augment=True)
    val_ds = KneeMRIDataset(X[val_idx], y[val_idx], augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = build_model()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device)
                outputs = torch.sigmoid(model(imgs)).cpu().numpy()
                val_preds.append(outputs)
                val_targets.append(labels.numpy())

        val_preds = np.concatenate(val_preds, axis=0)
        val_targets = np.concatenate(val_targets, axis=0)

        # Per-class AUC, skipping classes with only one label value present in this fold
        aucs = []
        for i in range(12):
            if len(np.unique(val_targets[:, i])) > 1:
                aucs.append(roc_auc_score(val_targets[:, i], val_preds[:, i]))
        macro_auc = np.mean(aucs) if aucs else float("nan")

        print(f"  Epoch {epoch+1}/{epochs} - train_loss: {train_loss/len(train_loader):.4f} - val_macro_auc: {macro_auc:.4f}")

    return macro_auc


# --- 5-fold cross-validation ---
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_aucs = []

for fold, (train_idx, val_idx) in enumerate(mskf.split(X, y)):
    print(f"\n=== Fold {fold+1}/5 ===")
    print(f"Train size: {len(train_idx)}, Val size: {len(val_idx)}")
    auc = train_one_fold(train_idx, val_idx, X, y)
    fold_aucs.append(auc)

print(f"\n=== Overall CV macro AUC: {np.mean(fold_aucs):.4f} (+/- {np.std(fold_aucs):.4f}) ===")

In [ ]:
!pip install sentence-transformers -q

## 5. Baseline: text-only classifier on the radiology reports
Multilingual sentence embeddings + per-label logistic regression. This cannot be the final model (test studies have no reports), but it shows how much signal the reports carry.

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

DATA_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train_df = pd.read_csv(os.path.join(DATA_ROOT, "train.csv"))

label_cols = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
              "Medial OA", "Lateral OA", "PF OA", "Effusion",
              "Synovitis", "Baker's", "Contusion", "Fracture"]

gold_df = train_df[train_df[label_cols].notna().all(axis=1)].reset_index(drop=True)
weak_df = train_df[train_df[label_cols].isna().all(axis=1)].reset_index(drop=True)

print(f"Gold: {len(gold_df)}, Weak (report-only): {len(weak_df)}")

# Load multilingual embedding model
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

# Embed all gold reports
gold_embeddings = embedder.encode(gold_df["Report"].tolist(), show_progress_bar=True)
y_gold = gold_df[label_cols].values.astype(np.float32)

print(f"Embedding shape: {gold_embeddings.shape}")  # should be (58, 768)

# Cross-validate a simple per-label logistic regression on report embeddings alone
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_aucs = []

for fold, (train_idx, val_idx) in enumerate(mskf.split(gold_embeddings, y_gold)):
    X_tr, X_val = gold_embeddings[train_idx], gold_embeddings[val_idx]
    y_tr, y_val = y_gold[train_idx], y_gold[val_idx]

    aucs = []
    for i, col in enumerate(label_cols):
        if len(np.unique(y_tr[:, i])) < 2:
            continue  # skip if this fold's training data has only one class for this label
        clf = LogisticRegression(max_iter=1000, class_weight="balanced")
        clf.fit(X_tr, y_tr[:, i])
        if len(np.unique(y_val[:, i])) < 2:
            continue
        preds = clf.predict_proba(X_val)[:, 1]
        aucs.append(roc_auc_score(y_val[:, i], preds))

    fold_auc = np.mean(aucs) if aucs else float("nan")
    fold_aucs.append(fold_auc)
    print(f"Fold {fold+1}: macro AUC = {fold_auc:.4f}")

print(f"\nText-only CV macro AUC: {np.mean(fold_aucs):.4f} (+/- {np.std(fold_aucs):.4f})")

### PCA / regularization sweep for the text classifier

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# gold_embeddings and y_gold already exist from the previous cell
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Try a few PCA component counts and regularization strengths to see what helps
configs = [
    {"n_components": 10, "C": 0.1},
    {"n_components": 20, "C": 0.1},
    {"n_components": 30, "C": 0.1},
    {"n_components": 20, "C": 0.01},
    {"n_components": 20, "C": 1.0},
]

for config in configs:
    n_components = config["n_components"]
    C = config["C"]
    fold_aucs = []

    for fold, (train_idx, val_idx) in enumerate(mskf.split(gold_embeddings, y_gold)):
        X_tr_raw, X_val_raw = gold_embeddings[train_idx], gold_embeddings[val_idx]
        y_tr, y_val = y_gold[train_idx], y_gold[val_idx]

        # Standardize before PCA
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr_raw)
        X_val_scaled = scaler.transform(X_val_raw)

        # PCA fit only on training fold to avoid leakage
        pca = PCA(n_components=n_components, random_state=42)
        X_tr = pca.fit_transform(X_tr_scaled)
        X_val = pca.transform(X_val_scaled)

        aucs = []
        for i in range(12):
            if len(np.unique(y_tr[:, i])) < 2:
                continue
            clf = LogisticRegression(max_iter=1000, class_weight="balanced", C=C)
            clf.fit(X_tr, y_tr[:, i])
            if len(np.unique(y_val[:, i])) < 2:
                continue
            preds = clf.predict_proba(X_val)[:, 1]
            aucs.append(roc_auc_score(y_val[:, i], preds))

        fold_aucs.append(np.mean(aucs) if aucs else float("nan"))

    print(f"n_components={n_components}, C={C}: macro AUC = {np.nanmean(fold_aucs):.4f} (+/- {np.nanstd(fold_aucs):.4f})")